In [2]:
# 1. 라이브러리 및 경로 설정

from pathlib import Path
import time
import random
import pandas as pd

from google_play_scraper import app, reviews, Sort

try:
    from google_play_scraper import search
    SEARCH_AVAILABLE = True
except ImportError:
    SEARCH_AVAILABLE = False

PROJECT_DIR = Path(r"C:\취준\mobile-game-retention-review-ua-analysis")

RAW_DIR = PROJECT_DIR / "data" / "raw"
COLLECTED_DIR = PROJECT_DIR / "data" / "collected"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_TABLE_DIR = PROJECT_DIR / "outputs" / "tables"
OUTPUT_FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
DOCS_DIR = PROJECT_DIR / "docs"
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"

for path in [
    RAW_DIR,
    COLLECTED_DIR,
    PROCESSED_DIR,
    OUTPUT_TABLE_DIR,
    OUTPUT_FIGURE_DIR,
    DOCS_DIR,
    NOTEBOOK_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Collected directory:", COLLECTED_DIR)
print("Search available:", SEARCH_AVAILABLE)

Project directory: C:\취준\mobile-game-retention-review-ua-analysis
Collected directory: C:\취준\mobile-game-retention-review-ua-analysis\data\collected
Search available: True


In [3]:
# 2. 수집 설정

LANG = "en"
COUNTRY = "us"

TARGET_APP_COUNT = 100
REVIEWS_PER_APP = 100
REVIEW_SORT = Sort.NEWEST

print("Target apps:", TARGET_APP_COUNT)
print("Reviews per app:", REVIEWS_PER_APP)

Target apps: 100
Reviews per app: 100


In [4]:
# 3. 검색 기반 앱 ID 수집 + 수동 보완

manual_app_ids = [
    "com.king.candycrushsaga",
    "com.supercell.clashofclans",
    "com.supercell.clashroyale",
    "com.supercell.brawlstars",
    "com.roblox.client",
    "com.nianticlabs.pokemongo",
    "com.vizorapps.klondike",
    "com.playrix.gardenscapes",
    "com.playrix.homescapes",
    "com.playrix.fishdomdd.gplay",
    "com.outfit7.mytalkingtomfree",
    "com.outfit7.mytalkingtom2",
    "com.outfit7.mytalkingangela2",
    "com.miniclip.eightballpool",
    "com.easybrain.sudoku.android",
    "com.peoplefun.wordcross",
    "com.tripledot.solitaire",
    "com.tripledot.woodoku",
    "com.halfbrick.fruitninjafree",
    "com.halfbrick.jetpackjoyride",
    "com.kiloo.subwaysurf",
    "com.imangi.templerun2",
    "com.rovio.baba",
    "com.rovio.angrybirds2",
    "com.innersloth.spacemafia",
    "com.bigduckgames.flow",
    "com.scopely.monopolygo",
    "com.ludo.king",
    "com.ea.game.pvzfree_row",
    "com.zynga.words3",
    "com.zynga.farmville3",
    "com.activision.callofduty.shooter",
    "com.tencent.ig",
    "com.mobile.legends",
    "jp.pokemon.pokemonunite",
]

search_queries = [
    "casual games",
    "hyper casual games",
    "arcade games",
    "puzzle games",
    "runner games",
    "idle games",
    "io games",
    "merge games",
    "simulation games",
    "mobile games",
]

app_ids = []

if SEARCH_AVAILABLE:
    for query in search_queries:
        try:
            results = search(
                query,
                lang=LANG,
                country=COUNTRY,
                n_hits=30
            )

            for item in results:
                app_id = item.get("appId")
                if app_id and app_id not in app_ids:
                    app_ids.append(app_id)

            print(f"Query '{query}' collected. Current app count: {len(app_ids)}")
            time.sleep(random.uniform(0.5, 1.0))

        except Exception as e:
            print(f"Search failed for query '{query}': {e}")

for app_id in manual_app_ids:
    if app_id not in app_ids:
        app_ids.append(app_id)

app_ids = app_ids[:TARGET_APP_COUNT]

print("Final app ids:", len(app_ids))
print(app_ids[:20])

Query 'casual games' collected. Current app count: 24
Query 'hyper casual games' collected. Current app count: 42
Query 'arcade games' collected. Current app count: 62
Query 'puzzle games' collected. Current app count: 80
Query 'runner games' collected. Current app count: 97
Query 'idle games' collected. Current app count: 127
Query 'io games' collected. Current app count: 153
Query 'merge games' collected. Current app count: 183
Query 'simulation games' collected. Current app count: 209
Query 'mobile games' collected. Current app count: 228
Final app ids: 100
['com.king.candycrushsaga', 'com.moonactive.coinmaster', 'com.JindoBlu.OfflineGames', 'com.king.candycrushsodasaga', 'com.block.juggle', 'com.playrix.fishdomdd.gplay', 'com.roblox.client', 'com.dreamgames.royalmatch', 'com.miniclip.carrom', 'com.merge.match3mystery', 'com.farlightgames.pgame.gp', 'com.superking.parchisi.star', 'com.miniclip.eightballpool', 'com.wildspike.wormszone', 'find.out.hidden.objects.seek.puzzle.games.free

In [5]:
# 4. 앱 메타데이터 수집
def safe_get_app_info(app_id, lang="en", country="us"):
    try:
        info = app(app_id, lang=lang, country=country)
        return {
            "app_id": app_id,
            "title": info.get("title"),
            "developer": info.get("developer"),
            "genre": info.get("genre"),
            "genre_id": info.get("genreId"),
            "score": info.get("score"),
            "ratings_count": info.get("ratings"),
            "reviews_count": info.get("reviews"),
            "installs": info.get("installs"),
            "min_installs": info.get("minInstalls"),
            "real_installs": info.get("realInstalls"),
            "free": info.get("free"),
            "price": info.get("price"),
            "currency": info.get("currency"),
            "contains_ads": info.get("containsAds"),
            "in_app_purchase": info.get("offersIAP"),
            "updated": info.get("updated"),
            "released": info.get("released"),
            "content_rating": info.get("contentRating"),
            "url": info.get("url"),
        }
    except Exception as e:
        return {
            "app_id": app_id,
            "error": str(e),
        }


app_info_rows = []

for i, app_id in enumerate(app_ids, start=1):
    row = safe_get_app_info(app_id, lang=LANG, country=COUNTRY)
    app_info_rows.append(row)

    print(f"[{i}/{len(app_ids)}] {app_id} - {row.get('title')}")
    time.sleep(random.uniform(0.3, 0.8))

apps_df = pd.DataFrame(app_info_rows)

display(apps_df.head())
print("Shape:", apps_df.shape)
print("Error count:", apps_df["error"].notna().sum() if "error" in apps_df.columns else 0)

apps_df.to_csv(
    COLLECTED_DIR / "google_play_apps.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", COLLECTED_DIR / "google_play_apps.csv")

[1/100] com.king.candycrushsaga - Candy Crush Saga
[2/100] com.moonactive.coinmaster - Coin Master
[3/100] com.JindoBlu.OfflineGames - Offline Games - No Wifi Games
[4/100] com.king.candycrushsodasaga - Candy Crush Soda Saga
[5/100] com.block.juggle - Block Blast!
[6/100] com.playrix.fishdomdd.gplay - Fishdom
[7/100] com.roblox.client - Roblox
[8/100] com.dreamgames.royalmatch - Royal Match
[9/100] com.miniclip.carrom - Carrom Pool: Disc Game
[10/100] com.merge.match3mystery - Mistfall Match: Design & Blast
[11/100] com.farlightgames.pgame.gp - Clash of Critters
[12/100] com.superking.parchisi.star - Parchisi STAR Online
[13/100] com.miniclip.eightballpool - 8 Ball Pool
[14/100] com.wildspike.wormszone - Worms Zone .io - Hungry Snake
[15/100] find.out.hidden.objects.seek.puzzle.games.free - Find It Out® - Hidden Object
[16/100] com.rubygames.assassin - Hunter Assassin
[17/100] com.king.candycrushjellysaga - Candy Crush Jelly Saga
[18/100] com.king.candycrush4 - Candy Crush Friends Saga

,app_id,title,developer,genre,genre_id,score,ratings_count,reviews_count,installs,min_installs,real_installs,free,price,currency,contains_ads,in_app_purchase,updated,released,content_rating,url
0,com.king.candycrushsaga,Candy Crush Saga,King,Casual,GAME_CASUAL,4.626835,38939374.0,2094860.0,"1,000,000,000+",1000000000,2261402010,True,0.0,USD,True,True,1.779963e+09,"Nov 15, 2012",Everyone,https://play.google.com/store/apps/details?id=...
1,com.moonactive.coinmaster,Coin Master,Moon Active,Casual,GAME_CASUAL,4.751316,10382177.0,691073.0,"100,000,000+",100000000,345137646,True,0.0,USD,True,True,1.780298e+09,"Feb 4, 2016",Teen,https://play.google.com/store/apps/details?id=...
2,com.JindoBlu.OfflineGames,Offline Games - No Wifi Games,JindoBlu,Casual,GAME_CASUAL,4.640068,457166.0,13820.0,"100,000,000+",100000000,192362829,True,0.0,USD,True,True,1.780866e+09,"Aug 4, 2023",Everyone,https://play.google.com/store/apps/details?id=...
3,com.king.candycrushsodasaga,Candy Crush Soda Saga,King,Casual,GAME_CASUAL,4.578125,8874939.0,390688.0,"500,000,000+",500000000,616471412,True,0.0,USD,True,True,1.779468e+09,"Nov 11, 2014",Everyone,https://play.google.com/store/apps/details?id=...
4,com.block.juggle,Block Blast!,HungryStudio,Puzzle,GAME_PUZZLE,4.820772,4769045.0,70652.0,"500,000,000+",500000000,969039804,True,0.0,USD,True,True,1.780662e+09,"Sep 23, 2022",Everyone,https://play.google.com/store/apps/details?id=...


Shape: (100, 20)
Error count: 0
Saved: C:\취준\mobile-game-retention-review-ua-analysis\data\collected\google_play_apps.csv


In [6]:
# 5. 수집 가능한 앱만 필터링
valid_apps_df = apps_df[
    apps_df["title"].notna()
].copy()

# 게임 장르만 최대한 유지
valid_apps_df = valid_apps_df[
    valid_apps_df["genre"].fillna("").str.contains("Game|Action|Arcade|Casual|Puzzle|Simulation|Strategy|Role Playing|Racing|Sports", case=False, regex=True)
    | valid_apps_df["genre_id"].fillna("").str.contains("GAME", case=False, regex=True)
].copy()

valid_app_ids = valid_apps_df["app_id"].dropna().unique().tolist()

print("Valid apps:", len(valid_app_ids))
display(valid_apps_df[["app_id", "title", "genre", "score", "installs", "contains_ads", "in_app_purchase"]].head(20))

valid_apps_df.to_csv(
    COLLECTED_DIR / "google_play_apps_valid.csv",
    index=False,
    encoding="utf-8-sig",
)

Valid apps: 100


,app_id,title,genre,score,installs,contains_ads,in_app_purchase
0,com.king.candycrushsaga,Candy Crush Saga,Casual,4.626835,"1,000,000,000+",True,True
1,com.moonactive.coinmaster,Coin Master,Casual,4.751316,"100,000,000+",True,True
2,com.JindoBlu.OfflineGames,Offline Games - No Wifi Games,Casual,4.640068,"100,000,000+",True,True
3,com.king.candycrushsodasaga,Candy Crush Soda Saga,Casual,4.578125,"500,000,000+",True,True
4,com.block.juggle,Block Blast!,Puzzle,4.820772,"500,000,000+",True,True
5,com.playrix.fishdomdd.gplay,Fishdom,Puzzle,4.604287,"100,000,000+",False,True
6,com.roblox.client,Roblox,Adventure,4.185744,"1,000,000,000+",False,True
7,com.dreamgames.royalmatch,Royal Match,Puzzle,4.635291,"100,000,000+",False,True
8,com.miniclip.carrom,Carrom Pool: Disc Game,Sports,4.483208,"500,000,000+",True,True
9,com.merge.match3mystery,Mistfall Match: Design & Blast,Casual,4.821782,"500,000+",True,True


In [7]:
# 6. 리뷰 수집 함수
def safe_collect_reviews(app_id, app_title=None, count=100, lang="en", country="us"):
    """
    앱별 최신 리뷰를 수집한다.
    실패 시 빈 리스트 반환.
    """
    try:
        result, continuation_token = reviews(
            app_id,
            lang=lang,
            country=country,
            sort=REVIEW_SORT,
            count=count,
        )

        rows = []
        for r in result:
            rows.append({
                "app_id": app_id,
                "title": app_title,
                "review_id": r.get("reviewId"),
                "review_text": r.get("content"),
                "review_score": r.get("score"),
                "review_date": r.get("at"),
                "thumbs_up_count": r.get("thumbsUpCount"),
                "review_created_version": r.get("reviewCreatedVersion"),
            })

        return rows

    except Exception as e:
        print(f"Review collection failed: {app_id} / {e}")
        return []

In [8]:
# 7. 앱별 리뷰 수집
all_review_rows = []

app_title_map = dict(zip(valid_apps_df["app_id"], valid_apps_df["title"]))

for i, app_id in enumerate(valid_app_ids, start=1):
    title = app_title_map.get(app_id)

    rows = safe_collect_reviews(
        app_id=app_id,
        app_title=title,
        count=REVIEWS_PER_APP,
        lang=LANG,
        country=COUNTRY,
    )

    all_review_rows.extend(rows)

    print(f"[{i}/{len(valid_app_ids)}] {title} ({app_id}) - reviews: {len(rows)} / total: {len(all_review_rows)}")

    # 너무 빠른 요청 방지
    time.sleep(random.uniform(0.5, 1.2))

reviews_df = pd.DataFrame(all_review_rows)

display(reviews_df.head())
print("Review shape:", reviews_df.shape)
print("Unique apps in reviews:", reviews_df["app_id"].nunique() if len(reviews_df) > 0 else 0)

reviews_df.to_csv(
    COLLECTED_DIR / "google_play_reviews.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", COLLECTED_DIR / "google_play_reviews.csv")

[1/100] Candy Crush Saga (com.king.candycrushsaga) - reviews: 100 / total: 100
[2/100] Coin Master (com.moonactive.coinmaster) - reviews: 100 / total: 200
[3/100] Offline Games - No Wifi Games (com.JindoBlu.OfflineGames) - reviews: 100 / total: 300
[4/100] Candy Crush Soda Saga (com.king.candycrushsodasaga) - reviews: 100 / total: 400
[5/100] Block Blast! (com.block.juggle) - reviews: 100 / total: 500
[6/100] Fishdom (com.playrix.fishdomdd.gplay) - reviews: 100 / total: 600
[7/100] Roblox (com.roblox.client) - reviews: 100 / total: 700
[8/100] Royal Match (com.dreamgames.royalmatch) - reviews: 100 / total: 800
[9/100] Carrom Pool: Disc Game (com.miniclip.carrom) - reviews: 100 / total: 900
[10/100] Mistfall Match: Design & Blast (com.merge.match3mystery) - reviews: 81 / total: 981
[11/100] Clash of Critters (com.farlightgames.pgame.gp) - reviews: 100 / total: 1081
[12/100] Parchisi STAR Online (com.superking.parchisi.star) - reviews: 100 / total: 1181
[13/100] 8 Ball Pool (com.miniclip

,app_id,title,review_id,review_text,review_score,review_date,thumbs_up_count,review_created_version
0,com.king.candycrushsaga,Candy Crush Saga,d07f84ae-3b1a-44bd-9c96-3c4895446156,I love a dopamine rush!,5,2026-06-07 18:06:18,0,1.329.0.1
1,com.king.candycrushsaga,Candy Crush Saga,515baacd-052f-455b-becf-f76da3522158,Unable to play after the new update.. Kindly f...,3,2026-06-07 18:05:58,0,1.329.0.1
2,com.king.candycrushsaga,Candy Crush Saga,359257e8-75f8-4728-9fad-84cf5509f135,Nice well game 👍,5,2026-06-07 18:05:29,0,NaN
3,com.king.candycrushsaga,Candy Crush Saga,05f385eb-d5bf-46c9-a9ad-cf853fa463b2,very interesting this game,4,2026-06-07 17:55:20,0,1.326.0.1
4,com.king.candycrushsaga,Candy Crush Saga,7b7804d9-6300-448f-ac34-2b29aa0f3642,nice game,5,2026-06-07 17:45:33,0,1.328.0.1


Review shape: (9761, 8)
Unique apps in reviews: 99
Saved: C:\취준\mobile-game-retention-review-ua-analysis\data\collected\google_play_reviews.csv


In [9]:
# 8. 수집 결과 품질 점검
collection_summary = {
    "target_app_count": TARGET_APP_COUNT,
    "collected_app_id_count": len(app_ids),
    "valid_app_count": len(valid_app_ids),
    "review_row_count": len(reviews_df),
    "review_app_count": reviews_df["app_id"].nunique() if len(reviews_df) > 0 else 0,
    "missing_review_text_count": reviews_df["review_text"].isna().sum() if len(reviews_df) > 0 else None,
    "missing_review_score_count": reviews_df["review_score"].isna().sum() if len(reviews_df) > 0 else None,
}

collection_summary_df = pd.DataFrame([collection_summary])
display(collection_summary_df)

collection_summary_df.to_csv(
    OUTPUT_TABLE_DIR / "google_play_collection_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# 앱별 리뷰 수
if len(reviews_df) > 0:
    app_review_count = (
        reviews_df.groupby(["app_id", "title"])
        .agg(
            review_count=("review_id", "count"),
            avg_review_score=("review_score", "mean"),
        )
        .reset_index()
        .sort_values("review_count", ascending=False)
    )

    display(app_review_count.head(20))

    app_review_count.to_csv(
        OUTPUT_TABLE_DIR / "google_play_app_review_count.csv",
        index=False,
        encoding="utf-8-sig",
    )

,target_app_count,collected_app_id_count,valid_app_count,review_row_count,review_app_count,missing_review_text_count,missing_review_score_count
0,100,100,100,9761,99,0,0


,app_id,title,review_count,avg_review_score
0,bubbleshooter.orig,Bubble Shooter - Classic Pop,100,4.24
1,com.JindoBlu.Antistress,Antistress - relaxation toys,100,4.74
2,com.JindoBlu.OfflineGames,Offline Games - No Wifi Games,100,4.58
3,com.Seriously.BestFiends,Best Fiends - Match 3 Puzzles,100,3.97
4,com.WalkTalk.FlightMaster,Epic Plane Evolution,100,3.52
5,com.bandagames.mpuzzle.gp,Magic Jigsaw Puzzles－Games HD,100,4.81
6,com.block.juggle,Block Blast!,100,4.64
7,com.codigames.hotel.empire.tycoon.idle.game,Hotel Empire Tycoon－Idle Game,100,4.00
8,com.crazylabs.lady.bug,Miraculous Ladybug & Cat Noir,100,4.16
9,com.creations.runnergame,Kooply Run™: Play and Create!,100,3.83


In [10]:
# 9. 리뷰 데이터 간단 전처리
if len(reviews_df) > 0:
    reviews_cleaned = reviews_df.copy()

    reviews_cleaned["review_text"] = reviews_cleaned["review_text"].fillna("").astype(str)
    reviews_cleaned["review_text_lower"] = reviews_cleaned["review_text"].str.lower()
    reviews_cleaned["review_date"] = pd.to_datetime(reviews_cleaned["review_date"], errors="coerce")
    reviews_cleaned["review_score"] = pd.to_numeric(reviews_cleaned["review_score"], errors="coerce")

    # 중복 제거
    before = len(reviews_cleaned)
    reviews_cleaned = reviews_cleaned.drop_duplicates(subset=["review_id"])
    after = len(reviews_cleaned)

    print("Before duplicates removed:", before)
    print("After duplicates removed:", after)
    print("Removed:", before - after)

    reviews_cleaned.to_csv(
        PROCESSED_DIR / "google_play_reviews_cleaned.csv",
        index=False,
        encoding="utf-8-sig",
    )

    display(reviews_cleaned.head())

else:
    print("No reviews collected.")

Before duplicates removed: 9761
After duplicates removed: 9761
Removed: 0


,app_id,title,review_id,review_text,review_score,review_date,thumbs_up_count,review_created_version,review_text_lower
0,com.king.candycrushsaga,Candy Crush Saga,d07f84ae-3b1a-44bd-9c96-3c4895446156,I love a dopamine rush!,5,2026-06-07 18:06:18,0,1.329.0.1,i love a dopamine rush!
1,com.king.candycrushsaga,Candy Crush Saga,515baacd-052f-455b-becf-f76da3522158,Unable to play after the new update.. Kindly f...,3,2026-06-07 18:05:58,0,1.329.0.1,unable to play after the new update.. kindly f...
2,com.king.candycrushsaga,Candy Crush Saga,359257e8-75f8-4728-9fad-84cf5509f135,Nice well game 👍,5,2026-06-07 18:05:29,0,NaN,nice well game 👍
3,com.king.candycrushsaga,Candy Crush Saga,05f385eb-d5bf-46c9-a9ad-cf853fa463b2,very interesting this game,4,2026-06-07 17:55:20,0,1.326.0.1,very interesting this game
4,com.king.candycrushsaga,Candy Crush Saga,7b7804d9-6300-448f-ac34-2b29aa0f3642,nice game,5,2026-06-07 17:45:33,0,1.328.0.1,nice game


In [11]:
# 10. 앱 메타데이터 간단 전처리
apps_cleaned = valid_apps_df.copy()

# 날짜 변환
apps_cleaned["updated"] = pd.to_datetime(apps_cleaned["updated"], unit="ms", errors="coerce")

# 숫자형 변환
numeric_cols = ["score", "ratings_count", "reviews_count", "min_installs", "real_installs", "price"]
for col in numeric_cols:
    if col in apps_cleaned.columns:
        apps_cleaned[col] = pd.to_numeric(apps_cleaned[col], errors="coerce")

# boolean 정리
bool_cols = ["free", "contains_ads", "in_app_purchase"]
for col in bool_cols:
    if col in apps_cleaned.columns:
        apps_cleaned[col] = apps_cleaned[col].astype("boolean")

apps_cleaned.to_csv(
    PROCESSED_DIR / "google_play_apps_cleaned.csv",
    index=False,
    encoding="utf-8-sig",
)

display(apps_cleaned.head())
print("Apps cleaned shape:", apps_cleaned.shape)

,app_id,title,developer,genre,genre_id,score,ratings_count,reviews_count,installs,min_installs,real_installs,free,price,currency,contains_ads,in_app_purchase,updated,released,content_rating,url
0,com.king.candycrushsaga,Candy Crush Saga,King,Casual,GAME_CASUAL,4.626835,38939374.0,2094860.0,"1,000,000,000+",1000000000,2261402010,True,0.0,USD,True,True,1970-01-21 14:26:03.225,"Nov 15, 2012",Everyone,https://play.google.com/store/apps/details?id=...
1,com.moonactive.coinmaster,Coin Master,Moon Active,Casual,GAME_CASUAL,4.751316,10382177.0,691073.0,"100,000,000+",100000000,345137646,True,0.0,USD,True,True,1970-01-21 14:31:37.778,"Feb 4, 2016",Teen,https://play.google.com/store/apps/details?id=...
2,com.JindoBlu.OfflineGames,Offline Games - No Wifi Games,JindoBlu,Casual,GAME_CASUAL,4.640068,457166.0,13820.0,"100,000,000+",100000000,192362829,True,0.0,USD,True,True,1970-01-21 14:41:05.641,"Aug 4, 2023",Everyone,https://play.google.com/store/apps/details?id=...
3,com.king.candycrushsodasaga,Candy Crush Soda Saga,King,Casual,GAME_CASUAL,4.578125,8874939.0,390688.0,"500,000,000+",500000000,616471412,True,0.0,USD,True,True,1970-01-21 14:17:48.032,"Nov 11, 2014",Everyone,https://play.google.com/store/apps/details?id=...
4,com.block.juggle,Block Blast!,HungryStudio,Puzzle,GAME_PUZZLE,4.820772,4769045.0,70652.0,"500,000,000+",500000000,969039804,True,0.0,USD,True,True,1970-01-21 14:37:41.750,"Sep 23, 2022",Everyone,https://play.google.com/store/apps/details?id=...


Apps cleaned shape: (100, 20)


In [12]:
# 11. 데이터 수집 문서 저장
collection_doc = f"""
# Google Play 게임 앱/리뷰 데이터 수집 요약

## 수집 목적
모바일 게임 앱마켓에서 관찰 가능한 유저 경험 신호를 분석하기 위해 Google Play의 게임 앱 메타데이터와 최신 리뷰를 수집했다.

## 수집 기준
- 국가: {COUNTRY}
- 언어: {LANG}
- 목표 앱 수: {TARGET_APP_COUNT}
- 앱별 목표 리뷰 수: {REVIEWS_PER_APP}

## 수집 결과
- 수집 앱 ID 수: {len(app_ids)}
- 유효 앱 수: {len(valid_app_ids)}
- 수집 리뷰 수: {len(reviews_df)}
- 리뷰가 수집된 앱 수: {reviews_df["app_id"].nunique() if len(reviews_df) > 0 else 0}

## 수집 컬럼

### 앱 메타데이터
- app_id
- title
- developer
- genre
- score
- ratings_count
- reviews_count
- installs
- free
- contains_ads
- in_app_purchase
- updated
- content_rating

### 리뷰 데이터
- app_id
- title
- review_id
- review_text
- review_score
- review_date
- thumbs_up_count
- review_created_version

## 해석 범위
이 데이터는 앱마켓에서 관찰 가능한 리뷰와 메타데이터 기반 유저 경험 신호를 분석하기 위한 데이터다.
개별 리뷰를 특정 UA 캠페인, 광고 채널, 유저 획득 비용과 직접 연결하지 않는다.
캠페인 KPI/ROAS 분석과는 독립 파트로 해석하며, 최종 종합 인사이트에서만 논리적으로 연결한다.
"""

doc_path = DOCS_DIR / "data_collection_plan.md"
doc_path.write_text(collection_doc, encoding="utf-8")

print("Saved:", doc_path)

Saved: C:\취준\mobile-game-retention-review-ua-analysis\docs\data_collection_plan.md
